In [1]:
import torch
import torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
print(model.fc)

total = sum(p.numel() for p in model.parameters())
print("total parameters:", total)

device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 172MB/s]


Linear(in_features=512, out_features=1000, bias=True)
total parameters: 11689512


In [2]:
import urllib.request, zipfile, os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
if not os.path.exists("hymenoptera_data"):
    urllib.request.urlretrieve(url, "data.zip")
    with zipfile.ZipFile("data.zip") as z:
        z.extractall(".")

normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize,
])

train_data = datasets.ImageFolder("hymenoptera_data/train", transform=train_tf)
val_data = datasets.ImageFolder("hymenoptera_data/val", transform=val_tf)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16)

print(train_data.classes)
print(len(train_data), len(val_data))

['ants', 'bees']
244 153


In [3]:
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(512, 2)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(model.fc)
print("trainable parameters:", trainable)
print("total parameters:", total)

Linear(in_features=512, out_features=2, bias=True)
trainable parameters: 1026
total parameters: 11177538


In [4]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

def accuracy(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            correct += (model(imgs).argmax(dim=1) == lbls).sum().item()
    return correct / len(loader.dataset)

print("before training: val accuracy", round(accuracy(model, val_loader), 3))

for epoch in range(5):
    model.train()
    running = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        loss = loss_fn(model(imgs), lbls)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item() * imgs.size(0)
    print(f"epoch {epoch+1}: train loss {running/len(train_data):.3f}, val accuracy {accuracy(model, val_loader):.3f}")

before training: val accuracy 0.497
epoch 1: train loss 0.701, val accuracy 0.837
epoch 2: train loss 0.423, val accuracy 0.928
epoch 3: train loss 0.336, val accuracy 0.961
epoch 4: train loss 0.291, val accuracy 0.961
epoch 5: train loss 0.231, val accuracy 0.967


In [5]:
scratch = models.resnet18(weights=None)
scratch.fc = nn.Linear(512, 2)
scratch = scratch.to(device)

optimizer = torch.optim.Adam(scratch.parameters(), lr=0.001)

for epoch in range(5):
    scratch.train()
    running = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        loss = loss_fn(scratch(imgs), lbls)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item() * imgs.size(0)
    print(f"epoch {epoch+1}: train loss {running/len(train_data):.3f}, val accuracy {accuracy(scratch, val_loader):.3f}")

epoch 1: train loss 1.145, val accuracy 0.575
epoch 2: train loss 0.717, val accuracy 0.575
epoch 3: train loss 0.705, val accuracy 0.608
epoch 4: train loss 0.675, val accuracy 0.699
epoch 5: train loss 0.545, val accuracy 0.699
